# Train NotAFigurine Classifier (Binary: writing_element vs figurine/noise)

Train a **binary classifier** to distinguish **writing elements** (letters, digits, punctuation)
from **non-writing elements** (figurines, blank areas, noise).

**Why binary?**
CV alone cannot distinguish a letter 'K' from figurine ♔ by shape—only by statistics.
This classifier learns: "does this glyph *look* like a normal character stroke pattern
vs. a specialized notation mark?" Helps filter false positives in figurine detection.

**Input:** Two zips from `extract_chess_glyphs`:
- `writing_elements_classifier.zip` — positive samples (letters/digits/punctuation from text)
- `figurine_classifier.zip` — negative samples (K, Q, R, B, N, blank)

**Output:** `notafigurine_classifier.tflite` — binary model (input: 1767-float HOG vector).

Architecture: same HOG+MLP as figurine_classifier, but 2-class instead of 5.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

## Step 2 — Copy and extract zips from Drive

Expects two zips on Drive (produced by `extract_chess_glyphs`):
- `writing_elements_classifier.zip` → positive class
- `figurine_classifier.zip` → negative class

In [ ]:
import os, shutil, zipfile

# ── EDIT THESE: paths to the zips on your Drive ────────────────────────
WRITING_ZIP_ON_DRIVE = '/content/gdrive/MyDrive/entrainement_ocr_echecs/writing_elements_classifier.zip'
FIGURINE_ZIP_ON_DRIVE = '/content/gdrive/MyDrive/entrainement_ocr_echecs/figurine_classifier.zip'

WRITING_LOCAL_ZIP = '/content/writing_elements_classifier.zip'
FIGURINE_LOCAL_ZIP = '/content/figurine_classifier.zip'
EXTRACT_DIR = '/content/glyphs_extracted'

# Check both zips exist
for zip_path, name in [
    (WRITING_ZIP_ON_DRIVE, 'writing_elements'),
    (FIGURINE_ZIP_ON_DRIVE, 'figurine'),
]:
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'{name} zip not found: {zip_path}')

# Copy both zips
print('Copying zips from Drive to local disk...')
shutil.copy2(WRITING_ZIP_ON_DRIVE, WRITING_LOCAL_ZIP)
shutil.copy2(FIGURINE_ZIP_ON_DRIVE, FIGURINE_LOCAL_ZIP)
print(f'  writing_elements: {os.path.getsize(WRITING_LOCAL_ZIP) / 1e6:.1f} MB')
print(f'  figurine:         {os.path.getsize(FIGURINE_LOCAL_ZIP) / 1e6:.1f} MB')

# Extract
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR)

print('\nExtracting...')
with zipfile.ZipFile(WRITING_LOCAL_ZIP, 'r') as z:
    z.extractall(os.path.join(EXTRACT_DIR, 'writing'))
with zipfile.ZipFile(FIGURINE_LOCAL_ZIP, 'r') as z:
    z.extractall(os.path.join(EXTRACT_DIR, 'figurine'))

# Locate glyphs/ folders
WRITING_DIR = None
FIGURINE_DIR = None

for root, dirs, files in os.walk(os.path.join(EXTRACT_DIR, 'writing')):
    if 'glyphs' in dirs:
        WRITING_DIR = os.path.join(root, 'glyphs')
        break
if WRITING_DIR is None and os.path.exists(os.path.join(EXTRACT_DIR, 'writing', 'glyphs')):
    WRITING_DIR = os.path.join(EXTRACT_DIR, 'writing', 'glyphs')

for root, dirs, files in os.walk(os.path.join(EXTRACT_DIR, 'figurine')):
    if 'glyphs' in dirs:
        FIGURINE_DIR = os.path.join(root, 'glyphs')
        break
if FIGURINE_DIR is None and os.path.exists(os.path.join(EXTRACT_DIR, 'figurine', 'glyphs')):
    FIGURINE_DIR = os.path.join(EXTRACT_DIR, 'figurine', 'glyphs')

if WRITING_DIR is None:
    raise FileNotFoundError(f'Could not locate writing glyphs/ folder')
if FIGURINE_DIR is None:
    raise FileNotFoundError(f'Could not locate figurine glyphs/ folder')

print(f'\n✅ writing elements: {WRITING_DIR}')
print(f'✅ figurines:        {FIGURINE_DIR}')

# Count images
from PIL import Image

def count_images(folder):
    count = 0
    for root, dirs, files in os.walk(folder):
        for f in files:
            path = os.path.join(root, f)
            try:
                Image.open(path).verify()
                count += 1
            except:
                pass
    return count

writing_count = count_images(WRITING_DIR)
figurine_count = count_images(FIGURINE_DIR)
print(f'\nPositive (writing): {writing_count} images')
print(f'Negative (figurine): {figurine_count} images')
print(f'Total: {writing_count + figurine_count} images')

## Step 3 — Install dependencies

In [ ]:
!pip install -q tensorflow pillow numpy scikit-image scikit-learn matplotlib

## Step 4 — Configuration and HOG feature extractor

Same HOG implementation as `figurine_classifier.ipynb` — must stay in sync with Dart.

In [ ]:
import gc
import ctypes
import random
import numpy as np
from PIL import Image, ImageOps, ImageFilter

CLASS_NAMES           = ['writing_element', 'not_writing_element']
IMG_SIZE              = 32
MAX_ROTATION_DEG      = 35
TARGET_TOTAL_SAMPLES  = 100_000
EPOCHS                = 80
BATCH_SIZE            = 128
VAL_SPLIT             = 0.15
TFLITE_PATH           = 'notafigurine_classifier.tflite'

# HOG constants — must stay in sync with hog_extractor.dart
_ORIENTATIONS = 9
_PX_PER_CELL  = 4
_CPB          = 2
_N_CELLS      = IMG_SIZE // _PX_PER_CELL          # 8
_N_BLOCKS     = _N_CELLS - _CPB + 1               # 7
_BLOCK_SIZE   = _CPB * _CPB * _ORIENTATIONS       # 36
FEATURE_DIM   = _N_BLOCKS * _N_BLOCKS * _BLOCK_SIZE + 3  # 1767

# Precompute pixel→cell mapping once
_CY   = np.repeat(np.arange(IMG_SIZE), IMG_SIZE) // _PX_PER_CELL
_CX   = np.tile(  np.arange(IMG_SIZE), IMG_SIZE) // _PX_PER_CELL
_BASE = (_CY * _N_CELLS + _CX) * _ORIENTATIONS

# malloc_trim: tell glibc to return freed memory to the OS (Linux/Colab only)
try:
    _libc = ctypes.CDLL('libc.so.6')
    _has_malloc_trim = True
    print('malloc_trim available ✅')
except Exception:
    _has_malloc_trim = False
    print('malloc_trim not available (non-Linux), GC only')


def _malloc_trim():
    gc.collect()
    if _has_malloc_trim:
        _libc.malloc_trim(0)


def _compute_hog_features(img_32x32: np.ndarray) -> np.ndarray:
    """Vectorized HOG — matches Dart HogExtractor.extract() exactly."""
    gx = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    gy = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    gx[:, 1:-1] = img_32x32[:, 2:] - img_32x32[:, :-2]
    gy[1:-1, :] = img_32x32[2:, :] - img_32x32[:-2, :]

    mag = np.sqrt(gx ** 2 + gy ** 2)
    ang = np.degrees(np.arctan2(gy, gx)) % 180.0

    bin_width = 180.0 / _ORIENTATIONS
    bf = ang.ravel() / bin_width
    b0 = bf.astype(np.int32) % _ORIENTATIONS
    b1 = (b0 + 1) % _ORIENTATIONS
    t  = bf - b0
    mf = mag.ravel()

    flat = np.bincount(
        np.concatenate([_BASE + b0, _BASE + b1]),
        weights=np.concatenate([mf * (1.0 - t), mf * t]),
        minlength=_N_CELLS * _N_CELLS * _ORIENTATIONS,
    )
    cell_hists = flat.reshape(_N_CELLS, _N_CELLS, _ORIENTATIONS)

    eps2    = 1e-5 ** 2
    hog_out = np.empty(_N_BLOCKS * _N_BLOCKS * _BLOCK_SIZE, dtype=np.float64)
    out_i   = 0
    for by in range(_N_BLOCKS):
        for bx in range(_N_BLOCKS):
            block = cell_hists[by:by + _CPB, bx:bx + _CPB, :].ravel().copy()
            block /= np.sqrt(np.dot(block, block) + eps2)
            np.clip(block, 0.0, 0.2, out=block)
            block /= np.sqrt(np.dot(block, block) + eps2)
            hog_out[out_i:out_i + _BLOCK_SIZE] = block
            out_i += _BLOCK_SIZE
    return hog_out


def extract_features(arr_32x32: np.ndarray, orig_w: int, orig_h: int) -> np.ndarray:
    """HOG + shape stats from a pre-resized 32×32 float32 numpy array."""
    img_gray = arr_32x32.astype(np.float64)
    hog_feat = _compute_hog_features(img_gray)
    extra = np.array([
        orig_w / max(orig_h, 1),
        np.mean(img_gray),
        np.std(img_gray),
    ], dtype=np.float64)
    return np.concatenate([hog_feat, extra]).astype(np.float32)


# Sanity-check
_dummy = extract_features(np.full((IMG_SIZE, IMG_SIZE), 0.5, dtype=np.float32), 32, 32)
assert len(_dummy) == FEATURE_DIM
del _dummy
mem_gb = TARGET_TOTAL_SAMPLES * FEATURE_DIM * 4 / 1e9
print(f'HOG feature vector : {FEATURE_DIM} dims  ✅')
print(f'Target samples     : {TARGET_TOTAL_SAMPLES:,}  (X ≈ {mem_gb:.2f} GB pre-allocated)')
print(f'Classes            : {len(CLASS_NAMES)}  {CLASS_NAMES}')

## Step 5 — Index and load writing elements (positive) and figurines (negative)

In [ ]:
def index_glyphs_recursive(root_dir):
    """Return list of (filepath, class_idx) — recursively scan all subdirs."""
    paths = []
    for root, dirs, files in os.walk(root_dir):
        for f in sorted(files):
            if f.lower().endswith('.png'):
                paths.append(os.path.join(root, f))
    return paths

print('Indexing glyphs...')
writing_paths = index_glyphs_recursive(WRITING_DIR)  # class 0
figurine_paths = index_glyphs_recursive(FIGURINE_DIR)  # class 1

print(f'✅ writing_element: {len(writing_paths)} images (positive)')
print(f'✅ not_writing_element: {len(figurine_paths)} images (negative)')
print(f'   Total: {len(writing_paths) + len(figurine_paths)} images')

## Step 6 — Data augmentation + HOG feature extraction

In [ ]:
def augment_to_array(pil_img, n, size=32):
    """Augment with rotation ±35°, scale, brightness, flip, noise."""
    orig_w, orig_h = pil_img.width, pil_img.height
    base_pil = pil_img.convert('L').resize((size, size), Image.LANCZOS)
    base = np.array(base_pil, dtype=np.float32) / 255.0
    del base_pil

    results = []
    for _ in range(n):
        arr = base.copy()

        # Brightness
        brightness = random.uniform(0.8, 1.2)
        arr = np.clip(arr * brightness, 0.0, 1.0).astype(np.float32)

        # Rotation
        pil = Image.fromarray((arr * 255).astype(np.uint8))
        pil = pil.rotate(random.uniform(-MAX_ROTATION_DEG, MAX_ROTATION_DEG),
                         resample=Image.BICUBIC, fillcolor=255)
        arr = np.array(pil, dtype=np.float32) / 255.0
        del pil

        # Scale
        scale    = random.uniform(0.80, 1.20)
        new_size = max(4, int(size * scale))
        pil2     = Image.fromarray((arr * 255).astype(np.uint8)).resize(
                       (new_size, new_size), Image.LANCZOS)
        small    = np.array(pil2, dtype=np.float32) / 255.0
        del pil2
        canvas   = np.ones((size, size), dtype=np.float32)
        off      = (size - new_size) // 2
        sy, sx   = max(0, off), max(0, off)
        ey       = min(sy + small.shape[0], size)
        ex       = min(sx + small.shape[1], size)
        canvas[sy:ey, sx:ex] = small[:ey - sy, :ex - sx]
        arr      = canvas

        # Flip
        if random.random() > 0.5:
            arr = arr[:, ::-1].copy()

        # Noise
        arr += np.random.normal(0, 0.03, arr.shape).astype(np.float32)
        results.append((np.clip(arr, 0.0, 1.0).astype(np.float32), orig_w, orig_h))
    return results


print(f'\n{"="*70}')
print('DATA AUGMENTATION (BALANCED BINARY)')
print(f'{"="*70}')

AUGMENT_TOTAL_PER_CLASS = 5000
KEEP_PER_CLASS = 25000

print(f'Per-class target  : {KEEP_PER_CLASS} images')
print(f'Augmentations gen : {AUGMENT_TOTAL_PER_CLASS} per class\n')

X_all, y_all = [], []

for class_idx, (class_name, paths) in enumerate([
    ('writing_element', writing_paths),
    ('not_writing_element', figurine_paths),
]):
    
    if not paths:
        print(f'⚠️  {class_name}: no images found')
        continue
    
    print(f'{class_name}: {len(paths)} originals')
    
    # Load originals and extract HOG
    X_orig, y_orig = [], []
    for i, filepath in enumerate(paths):
        try:
            with Image.open(filepath) as img:
                orig_w, orig_h = img.width, img.height
                arr = np.array(img.convert('L').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS), dtype=np.float32) / 255.0
            X_orig.append(extract_features(arr, orig_w, orig_h))
            y_orig.append(class_idx)
        except Exception as e:
            print(f'    Error loading {filepath}: {e}')
            continue
        
        if (i + 1) % max(1, len(paths) // 5) == 0:
            pct = 100 * (i + 1) // len(paths)
            print(f'  Originals: {pct:3d}%', flush=True)
    
    X_orig = np.array(X_orig, dtype=np.float32)
    
    # Generate augmentations
    num_to_augment = min(AUGMENT_TOTAL_PER_CLASS, len(paths) * 50)
    augment_per_image = max(1, num_to_augment // len(paths))
    
    X_aug, y_aug = [], []
    sampled_indices = random.sample(range(len(paths)), min(len(paths), AUGMENT_TOTAL_PER_CLASS // augment_per_image))
    
    aug_count = 0
    for idx, img_idx in enumerate(sampled_indices):
        filepath = paths[img_idx]
        try:
            with Image.open(filepath) as img:
                augs = augment_to_array(img, augment_per_image)
            for arr, orig_w, orig_h in augs:
                X_aug.append(extract_features(arr, orig_w, orig_h))
                y_aug.append(class_idx)
                aug_count += 1
        except Exception as e:
            print(f'    Error augmenting {filepath}: {e}')
            continue
        
        if (idx + 1) % max(1, len(sampled_indices) // 5) == 0:
            pct = 100 * (idx + 1) // len(sampled_indices)
            print(f'  Augmented: {pct:3d}% ({aug_count} samples)', flush=True)
    
    X_aug = np.array(X_aug, dtype=np.float32) if X_aug else np.empty((0, FEATURE_DIM), dtype=np.float32)
    
    # Combine
    X_combined = np.vstack([X_orig, X_aug])
    y_combined = np.array(y_orig + y_aug)
    
    # Shuffle
    perm = np.random.permutation(len(X_combined))
    X_combined = X_combined[perm]
    y_combined = y_combined[perm]
    
    # Keep first KEEP_PER_CLASS
    keep_count = min(KEEP_PER_CLASS, len(X_combined))
    X_combined = X_combined[:keep_count]
    y_combined = y_combined[:keep_count]
    
    X_all.append(X_combined)
    y_all.extend(y_combined)
    
    print(f'  → {len(X_orig)} orig + {len(X_aug)} aug = {len(X_combined)} total (kept {keep_count})\n')
    _malloc_trim()

# Concatenate
X = np.vstack(X_all)
y = np.array(y_all)

# Final shuffle
perm = np.random.permutation(len(X))
X = X[perm]
y = y[perm]

print(f'\n✅ Final dataset: {len(X):,} samples  X: {X.shape}  ({X.nbytes / 1e9:.2f} GB)')
for class_idx, class_name in enumerate(CLASS_NAMES):
    count = np.sum(y == class_idx)
    print(f'  {class_name}: {count:,}')

## Step 7 — Train MLP on HOG features

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, stratify=y, random_state=42
)
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(FEATURE_DIM,)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
], name='notafigurine_hog_mlp')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=12, restore_best_weights=True, monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=6, min_lr=1e-6, monitor='val_accuracy'),
    ],
    verbose=1,
)
print(f'\n✅ Best val accuracy: {max(history.history["val_accuracy"]):.4%}')

## Step 8 — Per-class accuracy report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

y_pred_probs = model.predict(X_val, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

print('Classification Report:')
print(classification_report(y_val, y_pred, target_names=CLASS_NAMES))

print('\nConfusion Matrix:')
cm = confusion_matrix(y_val, y_pred)
print(cm)

print('\nPer-class spot-check:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = y_val == cls_idx
    if mask.sum() == 0:
        continue
    probs = y_pred_probs[mask][0]
    pred  = CLASS_NAMES[np.argmax(probs)]
    print(f'  {"✅" if pred == cls_name else "⚠️ "} {cls_name} → {pred} ({probs.max():.4%})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'],         label='train')
ax2.plot(history.history['val_loss'],     label='val')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout(); plt.show()

## Step 9 — Export TFLite model

In [ ]:
converter    = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f'✅ Saved: {TFLITE_PATH} ({size_kb:.0f} KB)')

interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print(f'   Input : {inp["shape"]}  dtype={inp["dtype"].__name__}')
print(f'   Output: {out["shape"]}  dtype={out["dtype"].__name__}')

print('\nTFLite spot-check:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = y_val == cls_idx
    if mask.sum() == 0:
        continue
    sample = X_val[mask][0:1].astype(np.float32)
    interp.set_tensor(inp['index'], sample)
    interp.invoke()
    probs = interp.get_tensor(out['index'])[0]
    pred  = CLASS_NAMES[np.argmax(probs)]
    print(f'  {"✅" if pred == cls_name else "⚠️ "} {cls_name} → {pred} ({probs.max():.4%})')

## Step 10 — Download model

In [ ]:
from google.colab import files
files.download(TFLITE_PATH)
print(f'✅ Downloaded {TFLITE_PATH}')